# Fetch News Articles & Sentiment Indicators — Per-Ticker Pipeline

Single-ticker pipeline that produces everything needed for the paper replication in one run.
**Primary sentiment source: RavenPack (via WRDS).** Refinitiv is optional and only used to fetch raw article body text.

| Step | What it does |
|---|---|
| 1 | Connect to WRDS, look up RavenPack entity ID |
| 2 | Fetch all articles 2003–2014 (year-by-year, checkpointed) |
| 3 | Coverage statistics — passes paper filter if avg ≥ 1 article/week |
| 4 | (Optional) Fetch Refinitiv article text (saved separately) |
| 5 | Assemble final article-level dataset |
| 6 | Aggregate to weekly sentiment + buzz |
| 7 | Compute Relative Sentiment Indicators: Shock & Trend |
| 8 | Save all outputs |

**Set `TICKER` below and re-run for any stock.** RavenPack checkpoint makes Step 2 resumable.

---
### Output files (all in `data/raw/news/ravenpack/`, gitignored)

| File | Description |
|---|---|
| `{ticker}_articles_2003_2014.parquet` | One row per article: raw fields + sentiment_score |
| `{ticker}_daily_counts_2003_2014.csv` | Article count per calendar day |
| `{ticker}_weekly_sentiment_2003_2014.parquet` | Weekly avg sentiment, buzz |
| `{ticker}_rsi_features_2003_2014.parquet` | Shock, Trend, 1-week/1-month avg sentiment (LTR inputs) |
| `{ticker}_coverage_summary.json` | Coverage stats + RSI availability flag |
| `{ticker}_rf_article_text_2003_2014.parquet` | (Optional) Refinitiv article body text |
| `{ticker}_rp_checkpoint.parquet` + `{ticker}_rp_manifest.json` | Resumability checkpoint |

---
### Article-level dataset schema

**Raw fields** (from RavenPack via WRDS):

| Column | Type | Description |
|---|---|---|
| `article_time` | datetime (UTC) | Exact publication timestamp |
| `headline` | str | News headline text |
| `story_id` | str | RavenPack story identifier (`rp_story_id`) |
| `source_code` | str | News wire / source name |
| `article_date` | date | Eastern-time calendar date |
| `relevance_score` | float [0–1] | RavenPack relevance / 100 |
| `event_sentiment_score` | float [−1, +1] | RavenPack composite sentiment |

**Calculated fields**:

| Column | Formula |
|---|---|
| `sentiment_score` | `relevance_score × event_sentiment_score` — Eq. 8, Song et al. (2017) |
| `articles_that_day` | Total articles for this ticker on the same calendar date |

**Requires:** `WRDS_USERNAME` + `WRDS_PASSWORD` in `.env` (RavenPack). Optional: `LSEG_APP_KEY` for article text.

In [ ]:

from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import wrds
from dotenv import load_dotenv
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from sentiment_ltr.data.news_coverage import (
    build_news_query,
    daily_article_counts,
    fetch_headlines_for_window,
    month_chunks,
    quarter_chunks,
    summarize_news_coverage,
    weekly_article_counts,
    year_chunks,
)
from sentiment_ltr.data.refinitiv_queries import ticker_to_ric_candidates
from sentiment_ltr.data.refinitiv_session import open_refinitiv_session

# Refinitiv is optional — only needed for article text (Step 4)
try:
    import lseg.data as ld
    _LSEG_AVAILABLE = True
except ImportError:
    _LSEG_AVAILABLE = False
    print("lseg.data not installed — article text fetch (Step 4) will be skipped.")

print(f"Project root: {PROJECT_ROOT}")


In [ ]:

# ── Parameters ────────────────────────────────────────────────────────────────
TICKER  = "AAPL"
START   = "2003-01-01"
END     = "2014-12-31"
N_WEEKS = 8          # look-back window (weeks) for Shock & Trend

CHECKPOINT_EVERY  = 50    # story-text checkpoint cadence
STORY_FETCH_DELAY = 0.25  # seconds between Refinitiv get_story() calls

NEWS_DIR = PROJECT_ROOT / "data" / "raw" / "news" / "ravenpack"
NEWS_DIR.mkdir(parents=True, exist_ok=True)

slug = TICKER.lower()

# ── Final output files ────────────────────────────────────────────────────────
ARTICLES_PATH = NEWS_DIR / f"{slug}_articles_2003_2014.parquet"   # RavenPack articles + sentiment
DAILY_PATH    = NEWS_DIR / f"{slug}_daily_counts_2003_2014.csv"
WEEKLY_PATH   = NEWS_DIR / f"{slug}_weekly_sentiment_2003_2014.parquet"
RSI_PATH      = NEWS_DIR / f"{slug}_rsi_features_2003_2014.parquet"
SUMMARY_PATH  = NEWS_DIR / f"{slug}_coverage_summary.json"

# Optional Refinitiv article text (separate file — different story IDs from RavenPack)
RF_TEXT_PATH  = NEWS_DIR / f"{slug}_rf_article_text_2003_2014.parquet"

# ── Checkpoint files (safe to delete once the full run completes) ─────────────
RP_CHECKPOINT_PATH    = NEWS_DIR / f"{slug}_rp_checkpoint.parquet"
RP_MANIFEST_PATH      = NEWS_DIR / f"{slug}_rp_manifest.json"
STORY_CHECKPOINT_PATH = NEWS_DIR / f"{slug}_story_text_checkpoint.parquet"

print(f"Ticker : {TICKER}")
print(f"Window : {START} → {END}")
print(f"Output : {NEWS_DIR}")


In [ ]:

# ── Optional: open Refinitiv session for article text (Step 4) ────────────────
# RavenPack already provides the full sentiment pipeline.
# Refinitiv is only needed if you want to fetch the raw article body text.
# Note: Refinitiv uses its own story IDs (different from RavenPack's rp_story_id),
# so article text is stored in a separate file and not merged row-by-row.

HAVE_REFINITIV = False
if _LSEG_AVAILABLE:
    try:
        open_refinitiv_session(PROJECT_ROOT, ld)
        HAVE_REFINITIV = True
        print("Refinitiv session opened — article text fetch available.")
    except Exception as exc:
        print(f"Refinitiv unavailable (article text will be skipped): {exc}")
else:
    print("lseg.data not installed — skipping article text fetch.")



## Step 1 — Connect to WRDS and look up RavenPack entity

RavenPack assigns each company a stable `rp_entity_id`. We look it up once from the
mapping table so every year query can filter by it directly.


In [ ]:

db = wrds.Connection(
    wrds_username=os.environ["WRDS_USERNAME"],
    wrds_password=os.environ["WRDS_PASSWORD"],
)

def pg_sql(db: wrds.Connection, sql: str) -> pd.DataFrame:
    """Run SQL via raw psycopg2, bypassing SQLAlchemy 2.x incompatibility with wrds 3.x."""
    conn = db.connection.connection
    cur  = conn.cursor()
    cur.execute(sql)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    cur.close()
    return pd.DataFrame(rows, columns=cols)

mapping = pg_sql(db, f"""
    SELECT DISTINCT rp_entity_id, entity_name, ticker
    FROM ravenpack_common.wrds_rpa_company_mappings
    WHERE ticker = '{TICKER.upper()}'
""")

if mapping.empty:
    raise ValueError(f"No RavenPack entity found for {TICKER}. Check wrds_rpa_company_mappings.")

rp_entity_id = mapping["rp_entity_id"].iloc[0]
print(f"RavenPack entity ID : {rp_entity_id}")
print(f"Entity name         : {mapping['entity_name'].iloc[0]}")
print(mapping.to_string(index=False))



## Step 2 — Fetch RavenPack articles (WRDS, year-by-year)

Queries `ravenpack_dj.rpa_djpr_equities_{year}` for each year in the paper window.
Each completed year is saved to a checkpoint parquet immediately so the run is
fully resumable — re-running this cell skips any year already in the manifest.

**RavenPack → paper formula mapping:**

| RavenPack field | Role |
|---|---|
| `relevance` (0–100) | `relevance_score = relevance / 100` |
| `event_sentiment_score` (−1 to +1) | equivalent of `pos − neg` in TRNA |
| `sentiment_score` (computed) | `relevance_score × event_sentiment_score` — Eq. 8 |


In [ ]:

# "group" and "type" are SQL reserved words — quote them
RP_COLS = [
    "timestamp_utc", "rp_story_id", "relevance", "event_sentiment_score",
    "headline", "source_name", "topic", '"group"', '"type"', "news_type", "rpa_date_utc",
]

# ── Load checkpoint ───────────────────────────────────────────────────────────
if RP_CHECKPOINT_PATH.exists():
    rp_cache = pd.read_parquet(RP_CHECKPOINT_PATH)
    print(f"RavenPack checkpoint : {len(rp_cache):,} rows already saved.")
else:
    rp_cache = pd.DataFrame()

rp_completed: set[str] = set(json.loads(RP_MANIFEST_PATH.read_text())) if RP_MANIFEST_PATH.exists() else set()
if rp_completed:
    print(f"Completed years      : {sorted(rp_completed)}")

# ── Fetch year by year ────────────────────────────────────────────────────────
year_list = list(range(int(START[:4]), int(END[:4]) + 1))
cols_sql  = ", ".join(RP_COLS)

with tqdm(year_list, desc=f"{TICKER} RavenPack", unit="yr",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} yrs  [{elapsed}<{remaining}]") as pbar:
    for yr in pbar:
        yr_key = str(yr)
        pbar.set_postfix_str(yr_key)

        if yr_key in rp_completed:
            pbar.write(f"  {yr_key}: cached ✓")
            continue

        yr_start = max(START, f"{yr}-01-01")
        yr_end   = min(END,   f"{yr}-12-31")

        try:
            yr_df = pg_sql(db, f"""
                SELECT {cols_sql}
                FROM ravenpack_dj.rpa_djpr_equities_{yr}
                WHERE rp_entity_id = '{rp_entity_id}'
                  AND rpa_date_utc BETWEEN '{yr_start}' AND '{yr_end}'
                ORDER BY timestamp_utc
            """)
        except Exception as exc:
            pbar.write(f"  {yr_key}: ERROR — {exc}")
            continue

        pbar.write(f"  {yr_key}: {len(yr_df):,} rows")

        if not yr_df.empty:
            rp_cache = pd.concat([rp_cache, yr_df], ignore_index=True) if not rp_cache.empty else yr_df.copy()
            rp_cache.to_parquet(RP_CHECKPOINT_PATH, index=False)

        rp_completed.add(yr_key)
        RP_MANIFEST_PATH.write_text(json.dumps(sorted(rp_completed)))

print(f"\nTotal rows in cache : {len(rp_cache):,}")
print(f"Years completed     : {len(rp_completed)}/{len(year_list)}")


In [ ]:

# ── DIAGNOSTIC: test WRDS connection and table availability ───────────────────
# Run this cell if rp_cache is empty after the fetch cell completes.

# 1. Check connection is alive
try:
    pg_sql(db, "SELECT 1 AS ok")
    print("WRDS connection: OK")
except Exception as e:
    print(f"WRDS connection DEAD — re-run the discovery cell: {e}")

# 2. Test a single-year query for this entity
try:
    _test = pg_sql(db, f"""
        SELECT timestamp_utc, rp_story_id, relevance, event_sentiment_score, headline
        FROM ravenpack_dj.rpa_djpr_equities_2007
        WHERE rp_entity_id = '{rp_entity_id}'
        LIMIT 5
    """)
    print(f"\nTest query (2007, {TICKER}): {len(_test)} rows")
    print(_test.to_string())
except Exception as e:
    print(f"\nTest query failed: {e}")


In [ ]:

# ── Normalize to canonical schema + compute sentiment_score ───────────────────
rp = rp_cache.copy()

# Diagnose empty cache before anything else
if rp.empty or len(rp.columns) == 0:
    # Check whether the checkpoint file has data
    if RP_CHECKPOINT_PATH.exists():
        _ck = pd.read_parquet(RP_CHECKPOINT_PATH)
        print(f"Checkpoint file exists: {len(_ck):,} rows, columns: {_ck.columns.tolist()}")
        if not _ck.empty:
            rp = _ck.copy()
            rp_cache = _ck.copy()
            print("Loaded from checkpoint — continuing.")
        else:
            raise ValueError(
                "Checkpoint parquet is empty.\n"
                "Delete it and re-run the fetch cell (Step 2) to pull data from WRDS:\n"
                f"  rm '{RP_CHECKPOINT_PATH}'\n"
                f"  rm '{RP_MANIFEST_PATH}'"
            )
    else:
        raise ValueError(
            "rp_cache is empty — the RavenPack fetch cell (Step 2) has not run yet, "
            "or all year queries failed.\n"
            "Re-run the fetch cell and check for ERROR lines in its output."
        )

print("Available columns:", rp.columns.tolist())
print(f"Shape: {rp.shape}")

# Detect timestamp column — WRDS may return 'timestamp_utc' or 'rpa_date_utc'
if "timestamp_utc" in rp.columns:
    rp["article_time"] = pd.to_datetime(rp["timestamp_utc"], utc=True)
elif "rpa_date_utc" in rp.columns:
    rp["article_time"] = pd.to_datetime(rp["rpa_date_utc"], utc=True)
else:
    _ts_candidates = [c for c in rp.columns if "time" in c.lower() or "date" in c.lower()]
    if not _ts_candidates:
        raise KeyError(f"No timestamp column found. Columns: {rp.columns.tolist()}")
    rp["article_time"] = pd.to_datetime(rp[_ts_candidates[0]], utc=True)
    print(f"Used '{_ts_candidates[0]}' as timestamp column.")

rp["article_date"] = (
    rp["article_time"]
    .dt.tz_convert("America/New_York")
    .dt.normalize()
    .dt.tz_localize(None)
)
rp["week_start"] = rp["article_date"] - pd.to_timedelta(
    rp["article_date"].dt.dayofweek, unit="D"
)

rp = rp.rename(columns={"rp_story_id": "story_id", "source_name": "source_code"})

rp["relevance_score"]       = pd.to_numeric(rp["relevance"], errors="coerce") / 100
rp["event_sentiment_score"] = pd.to_numeric(rp["event_sentiment_score"], errors="coerce")
rp["sentiment_score"]       = rp["relevance_score"] * rp["event_sentiment_score"]

rp = rp.drop_duplicates(subset=["story_id"]).sort_values("article_time").reset_index(drop=True)
rp["ticker"] = TICKER.upper()
rp["source"] = "ravenpack_dj"

ARTICLE_COLS = [
    "ticker", "source",
    "article_time", "article_date", "week_start",
    "headline", "story_id", "source_code",
    "relevance_score", "event_sentiment_score", "sentiment_score",
    "topic", "group", "type", "news_type",
]
headlines = rp[[c for c in ARTICLE_COLS if c in rp.columns]]

print(f"Articles             : {len(headlines):,}")
print(f"Sentiment non-null   : {headlines['sentiment_score'].notna().sum():,}")
print(f"Date range           : {headlines['article_date'].min()} → {headlines['article_date'].max()}")
print(f"Sentiment score range: [{headlines['sentiment_score'].min():.3f}, {headlines['sentiment_score'].max():.3f}]")
headlines.head(3)


## Step 3 — Coverage Statistics

Checks whether this ticker passes the paper's news filter: **average ≥ 1 article per week** over the full window.

In [ ]:
daily   = daily_article_counts(headlines, START, END)
weekly  = weekly_article_counts(daily)

# Use rp_entity_id as ric placeholder for summary metadata
summary = summarize_news_coverage(TICKER, f"RP:{rp_entity_id}", headlines, START, END)

print(f"Total articles (unique)       : {summary.total_articles:,}")
print(f"Calendar days with ≥1 article : {summary.calendar_days_with_news:,}")
print(f"Weeks in range                : {summary.weeks_in_range:,}")
print(f"Weeks with zero articles      : {summary.weeks_with_zero_articles:,}")
print(f"Avg articles / week           : {summary.avg_articles_per_week:.2f}")
print(f"Passes paper threshold (≥1/wk): {summary.passes_paper_weekly_threshold}")

In [ ]:
fig = px.bar(
    daily[daily["article_count"] > 0],
    x="date", y="article_count",
    title=f"{TICKER} — Articles per day, {START} to {END}",
    labels={"date": "Date", "article_count": "Articles"},
    color_discrete_sequence=["#4C78A8"],
)
fig.update_layout(hovermode="closest", height=380)
fig.show()

In [ ]:
fig2 = px.line(
    weekly[weekly["article_count"] > 0],
    x="week_start", y="article_count",
    title=f"{TICKER} — Weekly article count, {START} to {END}",
    labels={"week_start": "Week", "article_count": "Articles"},
    color_discrete_sequence=["#4C78A8"],
)
fig2.update_traces(mode="lines+markers", marker={"size": 3})
fig2.update_layout(hovermode="closest", height=380)
fig2.show()


## Step 4 — (Optional) Fetch Refinitiv article text

This step is **independent of the sentiment pipeline** — RavenPack already provides
everything needed for Steps 5–7. Run this only if you need the raw article body text
(e.g. for future FinBERT experiments).

**Important:** Refinitiv uses its own story IDs (`story_id` starting with `urn:newsml:...`),
which are different from RavenPack's `rp_story_id`. The text is therefore saved in a
separate file (`{ticker}_rf_article_text_2003_2014.parquet`) and not merged into the
main article dataset.


In [ ]:

if not HAVE_REFINITIV:
    print("Refinitiv session not available — skipping article text fetch.")
    rf_headlines = pd.DataFrame()
else:
    # Fetch Refinitiv headlines to get their own story_ids for text fetching.
    # These are separate from RavenPack and use the adaptive year→quarter→month strategy.
    selected_ric = None
    for _ric in ticker_to_ric_candidates(TICKER):
        _q  = build_news_query(_ric)
        _s  = pd.Timestamp("2006-01-01").to_pydatetime()
        _e  = pd.Timestamp("2006-01-31").to_pydatetime()
        try:
            _raw = ld.news.get_headlines(_q, start=_s, end=_e, count=10)
            if _raw is not None:
                _df = _raw if isinstance(_raw, pd.DataFrame) else getattr(_raw, "data", None)
                if _df is not None and not _df.empty:
                    selected_ric = _ric
                    break
        except Exception:
            continue

    if selected_ric is None:
        print("Could not resolve a Refinitiv RIC — skipping article text fetch.")
        HAVE_REFINITIV = False
        rf_headlines = pd.DataFrame()
    else:
        print(f"Refinitiv RIC: {selected_ric}")

        # Load existing story-text checkpoint
        if STORY_CHECKPOINT_PATH.exists():
            story_checkpoint = pd.read_parquet(STORY_CHECKPOINT_PATH)
            already_done     = set(story_checkpoint["story_id"].dropna().astype(str))
            print(f"Story checkpoint: {len(already_done):,} stories already fetched.")
        else:
            story_checkpoint = pd.DataFrame()
            already_done     = set()

        # Fetch Refinitiv headlines to get story_ids
        print("Fetching Refinitiv headlines for story IDs...")
        rf_query   = build_news_query(selected_ric)
        rf_chunks  = year_chunks(START, END)
        rf_frames: list[pd.DataFrame] = []

        for yr_s, yr_e in rf_chunks:
            try:
                fr = fetch_headlines_for_window(
                    ld, rf_query,
                    yr_s.strftime("%Y-%m-%d"), yr_e.strftime("%Y-%m-%d"),
                    max_headlines=10_000,
                )
                if not fr.empty:
                    rf_frames.append(fr)
            except Exception:
                continue

        rf_headlines = pd.concat(rf_frames, ignore_index=True).drop_duplicates(
            subset=["storyId"] if "storyId" in pd.concat(rf_frames).columns else ["article_time", "headline"]
        ) if rf_frames else pd.DataFrame()

        if rf_headlines.empty:
            print("No Refinitiv headlines found.")
        else:
            # Normalise story_id column name
            if "storyId" in rf_headlines.columns:
                rf_headlines = rf_headlines.rename(columns={"storyId": "story_id"})
            pending = rf_headlines.dropna(subset=["story_id"])
            pending = pending[~pending["story_id"].astype(str).isin(already_done)]
            print(f"Refinitiv headlines : {len(rf_headlines):,}")
            print(f"Pending text fetch  : {len(pending):,}")


In [ ]:

if not HAVE_REFINITIV or rf_headlines.empty:
    print("Skipping article text fetch (Refinitiv unavailable or no headlines).")
else:
    fetched_rows: list[dict] = []
    errors_count = 0

    with tqdm(pending.iterrows(), total=len(pending), unit="story",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} stories  [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
        for i, (_, row) in enumerate(pbar, start=1):
            sid  = str(row["story_id"]).strip()
            text = None
            try:
                raw  = ld.news.get_story(sid, format=ld.news.Format.TEXT)
                text = str(raw) if raw is not None else None
            except Exception as exc:
                errors_count += 1
                if errors_count <= 3:
                    pbar.write(f"  [warn] story_id {sid}: {exc}")

            fetched_rows.append({"story_id": sid, "article_text": text})
            pbar.set_postfix(errors=errors_count)

            if i % CHECKPOINT_EVERY == 0 or i == len(pending):
                batch            = pd.DataFrame(fetched_rows)
                story_checkpoint = pd.concat([story_checkpoint, batch], ignore_index=True) if not story_checkpoint.empty else batch
                story_checkpoint.to_parquet(STORY_CHECKPOINT_PATH, index=False)
                fetched_rows = []
                pbar.write(f"  checkpoint saved at {i:,}/{len(pending):,} ({errors_count} errors)")

            if STORY_FETCH_DELAY > 0:
                time.sleep(STORY_FETCH_DELAY)

    print(f"\nDone. {len(pending):,} fetched, {errors_count:,} errors.")

    # Save Refinitiv article text as a separate file (story IDs differ from RavenPack)
    if not story_checkpoint.empty:
        rf_text = (
            rf_headlines
            .merge(story_checkpoint[["story_id", "article_text"]], on="story_id", how="left")
        )
        rf_text["ticker"] = TICKER.upper()
        rf_text.to_parquet(RF_TEXT_PATH, index=False)
        print(f"Refinitiv text saved → {RF_TEXT_PATH.name}  ({len(rf_text):,} rows)")


## Step 5 — Assemble Final Article Dataset

`headlines` (from RavenPack) already contains `sentiment_score = relevance_score × event_sentiment_score` (Eq. 8).

This step adds the `articles_that_day` count field and fixes up column ordering.
The optional Refinitiv article text — fetched in Step 4 — is saved separately because
RavenPack and Refinitiv use incompatible story ID schemes.

In [ ]:

# ── Add articles_that_day count ────────────────────────────────────────────────
daily_lookup = (
    daily
    .rename(columns={"article_count": "articles_that_day"})
    .assign(date=lambda d: pd.to_datetime(d["date"]).dt.normalize())
)
articles = headlines.copy()
articles["article_date"] = pd.to_datetime(articles["article_date"]).dt.normalize()
articles = articles.merge(daily_lookup, left_on="article_date", right_on="date", how="left").drop(columns=["date"])

# ── Final column order ─────────────────────────────────────────────────────────
COL_ORDER = [
    "ticker", "source",
    "article_time", "article_date", "week_start",
    "headline", "story_id", "source_code",
    "relevance_score", "event_sentiment_score", "sentiment_score",
    "topic", "group", "type", "news_type",
    "articles_that_day",
]
articles = articles[[c for c in COL_ORDER if c in articles.columns]].sort_values("article_time").reset_index(drop=True)

print(f"Article dataset       : {len(articles):,} rows")
print(f"With sentiment_score  : {articles['sentiment_score'].notna().sum():,}")
articles.head(3)


## Step 6 — Weekly Sentiment Aggregation

Average `sentiment_score` and buzz count per ISO week. Missing weeks (no news) are filled with `NaN` sentiment and `0` buzz.

In [ ]:
all_weeks = pd.date_range(START, END, freq="W-MON").normalize()

weekly_agg = (
    articles.groupby("week_start", as_index=False)
    .agg(
        sentiment_score=("sentiment_score", "mean"),
        buzz=("sentiment_score", "count"),
        avg_relevance=("relevance_score", "mean"),
    )
    .set_index("week_start")
    .reindex(all_weeks)
    .rename_axis("week_start")
    .reset_index()
)

weekly_agg["buzz"]   = weekly_agg["buzz"].fillna(0).astype(int)
weekly_agg["ticker"] = TICKER.upper()
weekly_agg["source"] = "ravenpack_dj"

print(f"Weekly rows     : {len(weekly_agg):,}")
print(f"Weeks with news : {(weekly_agg['buzz'] > 0).sum():,}")
weekly_agg.head(8)

## Step 7 — Relative Sentiment Indicators

Computes the four sentiment-based features used by the paper's learning-to-rank model.
All use **lagged** values only — no look-ahead.

| Feature | Formula (Song et al. 2017) |
|---|---|
| `shock` | `(S(t) − μ_{t−N…t−1}) / σ_{t−N…t−1}` — Eq. 1 |
| `trend` | `Σ ΔS(i)` for `i ∈ [t−N, t−1]` — Eq. 2 |
| `avg_sentiment_1w` | `S(t−1)` |
| `avg_sentiment_1m` | `mean(S(t−4)…S(t−1))` |

In [ ]:
rsi = weekly_agg.sort_values("week_start").reset_index(drop=True).copy()
s   = rsi["sentiment_score"]

roll_mean = s.shift(1).rolling(N_WEEKS, min_periods=max(1, N_WEEKS // 2)).mean()
roll_std  = s.shift(1).rolling(N_WEEKS, min_periods=max(2, N_WEEKS // 2)).std(ddof=1)

rsi["shock"]            = (s - roll_mean) / roll_std.replace(0, np.nan)
rsi["trend"]            = s.diff().shift(1).rolling(N_WEEKS, min_periods=1).sum()
rsi["avg_sentiment_1w"] = s.shift(1)
rsi["avg_sentiment_1m"] = s.shift(1).rolling(4, min_periods=1).mean()
rsi["n_weeks"]          = N_WEEKS

RSI_COLS = ["week_start", "ticker", "source",
            "sentiment_score", "buzz",
            "shock", "trend", "avg_sentiment_1w", "avg_sentiment_1m",
            "n_weeks"]
rsi_features = rsi[[c for c in RSI_COLS if c in rsi.columns]]

non_null = rsi_features[["shock", "trend"]].dropna().shape[0]
print(f"RSI feature rows (non-null): {non_null:,} / {len(rsi_features):,}")
rsi_features.dropna(subset=["shock", "trend"]).head(6)

In [ ]:
import matplotlib.pyplot as plt

if rsi_features["sentiment_score"].notna().any():
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    axes[0].plot(rsi_features["week_start"], rsi_features["sentiment_score"], lw=0.8)
    axes[0].axhline(0, color="grey", lw=0.5, ls="--")
    axes[0].set_ylabel("Weekly sentiment")
    axes[0].set_title(f"{TICKER} — Sentiment Indicators (RavenPack, N={N_WEEKS} weeks)")

    axes[1].plot(rsi_features["week_start"], rsi_features["shock"], lw=0.8, color="tab:orange")
    axes[1].axhline(0, color="grey", lw=0.5, ls="--")
    axes[1].set_ylabel("Shock (z-score)")

    axes[2].plot(rsi_features["week_start"], rsi_features["trend"], lw=0.8, color="tab:green")
    axes[2].axhline(0, color="grey", lw=0.5, ls="--")
    axes[2].set_ylabel("Trend (Σ ΔS)")

    fig.tight_layout()
    plt.show()
else:
    print("No sentiment data to plot.")

## Step 8 — Save Outputs

In [ ]:
articles.to_parquet(ARTICLES_PATH, index=False)
daily.to_csv(DAILY_PATH, index=False)
weekly_agg.to_parquet(WEEKLY_PATH, index=False)
rsi_features.to_parquet(RSI_PATH, index=False)

summary_dict = {
    "created_at":                    datetime.now(timezone.utc).isoformat(),
    "ticker":                        summary.ticker,
    "rp_entity_id":                  rp_entity_id,
    "sentiment_source":              "ravenpack_dj",
    "start_date":                    summary.start_date,
    "end_date":                      summary.end_date,
    "total_articles":                summary.total_articles,
    "calendar_days_with_news":       summary.calendar_days_with_news,
    "calendar_days_in_range":        summary.calendar_days_in_range,
    "avg_articles_per_calendar_day": round(summary.avg_articles_per_calendar_day, 4),
    "avg_articles_per_week":         round(summary.avg_articles_per_week, 4),
    "weeks_in_range":                summary.weeks_in_range,
    "weeks_with_zero_articles":      summary.weeks_with_zero_articles,
    "passes_paper_weekly_threshold": summary.passes_paper_weekly_threshold,
    "articles_with_sentiment":       int(articles["sentiment_score"].notna().sum()),
    "rsi_non_null_weeks":            int(rsi_features[["shock", "trend"]].dropna().shape[0]),
    "n_weeks_lookback":              N_WEEKS,
    "refinitiv_text_fetched":        HAVE_REFINITIV and RF_TEXT_PATH.exists(),
    "outputs": {
        "articles":       str(ARTICLES_PATH.relative_to(PROJECT_ROOT)),
        "daily_csv":      str(DAILY_PATH.relative_to(PROJECT_ROOT)),
        "weekly":         str(WEEKLY_PATH.relative_to(PROJECT_ROOT)),
        "rsi":            str(RSI_PATH.relative_to(PROJECT_ROOT)),
        "rp_checkpoint":  str(RP_CHECKPOINT_PATH.relative_to(PROJECT_ROOT)),
    },
}
if HAVE_REFINITIV and RF_TEXT_PATH.exists():
    summary_dict["outputs"]["rf_article_text"] = str(RF_TEXT_PATH.relative_to(PROJECT_ROOT))

SUMMARY_PATH.write_text(json.dumps(summary_dict, indent=2) + "\n", encoding="utf-8")

print(f"articles parquet → {ARTICLES_PATH.name}")
print(f"daily CSV        → {DAILY_PATH.name}")
print(f"weekly parquet   → {WEEKLY_PATH.name}")
print(f"RSI parquet      → {RSI_PATH.name}")
print(f"summary JSON     → {SUMMARY_PATH.name}")
print()
print("Load in another notebook:")
print(f"  articles = pd.read_parquet(PROJECT_ROOT / 'data/raw/news/ravenpack/{slug}_articles_2003_2014.parquet')")
print(f"  rsi      = pd.read_parquet(PROJECT_ROOT / 'data/raw/news/ravenpack/{slug}_rsi_features_2003_2014.parquet')")

In [ ]:
# Close WRDS connection
try:
    db.close()
    print("WRDS connection closed.")
except Exception as exc:
    print(f"WRDS close (safe to ignore): {exc}")

# Close Refinitiv session if it was opened
if HAVE_REFINITIV:
    try:
        ld.close_session()
        print("Refinitiv session closed.")
    except Exception as exc:
        print(f"Refinitiv close (safe to ignore): {exc}")